# 05 · Verificación final
Última tarea del job. Ejecuta comprobaciones sobre las tablas publicadas; **cada una debe devolver 0 filas incorrectas**.
Si alguna falla, la tarea lanza una excepción y el job queda en estado *Failed*.

In [0]:
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
from pyspark.sql import functions as F
from src import config as C

dbutils.widgets.text("catalogo", C.CATALOGO_POR_DEFECTO)
dbutils.widgets.text("lote", "lote_01")
cat, lote = dbutils.widgets.get("catalogo"), dbutils.widgets.get("lote")
t = lambda nombre: f"{cat}.{nombre}"

from src.verificaciones import ejecutar

In [0]:
resultados = ejecutar(spark, cat, lote)
fallos = [r[0] for r in resultados if r[2] != "OK"]

display(spark.createDataFrame(resultados, "comprobacion string, filas_incorrectas long, estado string"))

## Resumen del estado tras este lote

In [0]:
display(spark.sql(f"""
SELECT 'bronze.envios_raw' AS tabla, count(*) AS filas FROM {t(C.T_BRONZE)}
UNION ALL SELECT 'silver.envios_limpios', count(*) FROM {t(C.T_SILVER)}
UNION ALL SELECT 'silver.envios_rechazados', count(*) FROM {t(C.T_RECHAZOS)}
UNION ALL SELECT 'gold.dim_cliente (versiones)', count(*) FROM {t(C.T_DIM_CLIENTE)}
UNION ALL SELECT 'gold.dim_cliente (vigentes)', count(*) FROM {t(C.T_DIM_CLIENTE)} WHERE es_vigente
UNION ALL SELECT 'gold.fact_envios', count(*) FROM {t(C.T_FACT)}"""))
display(spark.table(t(C.T_LOG)).filter(F.col("lote_id") == lote).orderBy("tabla"))

In [0]:
if fallos:
    raise AssertionError("Verificación fallida: " + "; ".join(fallos))
print("Todas las comprobaciones han pasado.")